In [ ]:
@

In [ ]:
!pip install deepxde torch numpy matplotlib

In [ ]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt
import torch

# --- Config ---
ALPHA      = 1.0    # thermal diffusivity
T_MAX      = 1.0    # simulation end time
N_DOMAIN   = 10000  # interior collocation points (LHS)
N_BOUNDARY = 800    # spatial boundary points (200 per edge)
N_INITIAL  = 500    # IC points at t=0
W_PDE = 1           # loss weight: PDE residual
W_BC  = 10          # loss weight: boundary condition
W_IC  = 10          # loss weight: initial condition
T_EVAL = [0.25, 0.50, 0.75]  # time snapshots for evaluation

print(f"Backend: {dde.backend.backend_name}")
print("Config loaded.")

In [ ]:
def T_exact(x, y, t):
    """Exact solution: T = sin(πx)·sin(πy)·exp(−2π²αt)"""
    return np.sin(np.pi * x) * np.sin(np.pi * y) * np.exp(-2 * np.pi**2 * ALPHA * t)

# Sanity check: at t=0, T_exact should equal the IC
x_test = np.array([0.5])
y_test = np.array([0.5])
val_at_t0 = T_exact(x_test, y_test, 0.0)
val_ic    = np.sin(np.pi * 0.5) * np.sin(np.pi * 0.5)
print(f"T_exact(0.5, 0.5, 0) = {val_at_t0[0]:.6f}")
print(f"sin(π·0.5)²          = {val_ic:.6f}")
assert np.isclose(val_at_t0[0], val_ic), "Analytical solution does not match IC"
print("Sanity check passed.")

In [ ]:
# Spatial domain: unit square [0,1] x [0,1]
geom = dde.geometry.Rectangle(xmin=[0, 0], xmax=[1, 1])

# Time domain: [0, T_MAX]
timedomain = dde.geometry.TimeDomain(0, T_MAX)

# Combined space-time domain
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

print(f"Time domain: [0, {T_MAX}]")
print("GeometryXTime created successfully.")

In [ ]:
def pde(x, u):
    """
    Residual of: ∂u/∂t - α(∂²u/∂x² + ∂²u/∂y²) = 0
    x[:, 0] = x_coord, x[:, 1] = y_coord, x[:, 2] = t
    """
    u_t  = dde.grad.jacobian(u, x, i=0, j=2)          # ∂u/∂t
    u_xx = dde.grad.hessian(u, x, component=0, i=0, j=0)  # ∂²u/∂x²
    u_yy = dde.grad.hessian(u, x, component=0, i=1, j=1)  # ∂²u/∂y²
    return u_t - ALPHA * (u_xx + u_yy)

print("PDE residual function defined.")

In [ ]:
# Dirichlet BC: T = 0 on all spatial boundaries for all t
def on_boundary(x, on_boundary):
    return on_boundary

bc = dde.icbc.DirichletBC(
    geomtime,
    func=lambda x: np.zeros((len(x), 1)),
    on_boundary=on_boundary,
)

# IC: T(x, y, 0) = sin(πx)·sin(πy)
def ic_func(x):
    return np.sin(np.pi * x[:, 0:1]) * np.sin(np.pi * x[:, 1:2])

ic = dde.icbc.IC(
    geomtime,
    func=ic_func,
    on_initial=lambda x, on_initial: on_initial,
)

print("Boundary condition (Dirichlet, T=0) defined.")
print("Initial condition (sin(πx)·sin(πy)) defined.")

In [ ]:
data = dde.data.TimePDE(
    geometryxtime=geomtime,
    pde=pde,
    ic_bcs=[bc, ic],
    num_domain=N_DOMAIN,
    num_boundary=N_BOUNDARY,
    num_initial=N_INITIAL,
    train_distribution="LHS",
)

print("Training dataset assembled:")
print(f"  Domain points : {N_DOMAIN}")
print(f"  Boundary pts  : {N_BOUNDARY}")
print(f"  IC points     : {N_INITIAL}")

In [ ]:
# FNN: [3, 32, 32, 32, 32, 32, 1] with tanh activations
layer_sizes = [3] + [32] * 5 + [1]
net = dde.nn.FNN(layer_sizes, "tanh", "Glorot normal")

model = dde.Model(data, net)

total_params = sum(p.numel() for p in net.parameters())
print(f"Network: {layer_sizes}")
print(f"Activation: tanh")
print(f"Total parameters: {total_params:,}")

In [ ]:
model.compile(
    "adam",
    lr=1e-3,
    loss_weights=[W_PDE, W_BC, W_IC],
)

losshistory, train_state = model.train(iterations=10000)

dde.utils.plot_loss_history(losshistory)
plt.title("Adam training loss")
plt.tight_layout()
plt.show()
print("Adam training complete.")

In [ ]:
model.compile("L-BFGS", loss_weights=[W_PDE, W_BC, W_IC])
losshistory, train_state = model.train()

dde.utils.plot_loss_history(losshistory)
plt.title("L-BFGS fine-tuning loss")
plt.tight_layout()
plt.show()
print("L-BFGS fine-tuning complete.")